# Structure

Size, connectivity, diameter and radius for the three periods.
Done twice: all edges, and only edges with weight >= 2 (pairs that met more than once).

In [33]:
from pathlib import Path

import networkx as nx
import pandas as pd

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

nodes = pd.read_csv(DATA_DIR / "nodes.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")
names = dict(zip(nodes["id"], nodes["name"]))
PERIODS = ["sunak", "starmer", "burnham"]   # Prime Minister in office
print(len(nodes), "MPs,", len(edges), "edge rows")

994 MPs, 103367 edge rows


### Graphs

Directed, one edge per pair with the count as weight. With w >= 2 the MPs who lose all their edges stay as isolated nodes.

In [34]:
def build_graph(period, min_weight=1):
    period_edges = edges[edges["period"] == period]
    people = set(period_edges["source"]) | set(period_edges["target"])

    G = nx.DiGraph()
    for _, row in nodes[nodes["id"].isin(people)].iterrows():
        G.add_node(row["id"], name=row["name"], party=row[f"party_{period}"])
    kept = period_edges[period_edges["weight"] >= min_weight]
    G.add_weighted_edges_from(kept[["source", "target", "weight"]].itertuples(index=False))
    return G

graphs = {
    (period, w): build_graph(period, w)
    for period in PERIODS
    for w in [1, 2]
}
for (period, w), G in graphs.items():
    print(f"{period:6} weight>={w}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

sunak  weight>=1: 638 nodes, 35739 edges
sunak  weight>=2: 638 nodes, 13486 edges
starmer weight>=1: 643 nodes, 59528 edges
starmer weight>=2: 643 nodes, 23196 edges
burnham weight>=1: 605 nodes, 8100 edges
burnham weight>=2: 605 nodes, 1358 edges


### Size and connectivity

In [35]:
def structure(G):
    scc = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
    wcc = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
    return {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "total weight": int(G.size(weight="weight")),
        "density": round(nx.density(G), 4),
        "strongly connected": nx.is_strongly_connected(G),
        "weakly connected": nx.is_weakly_connected(G),
        "strong components": len(scc),
        "largest strong component": len(scc[0]),
        "weak components": len(wcc),
        "largest weak component": len(wcc[0]),
        "isolated MPs": nx.number_of_isolates(G),
    }

table = pd.DataFrame({f"{p} (w>={w})": structure(G) for (p, w), G in graphs.items()})
table

,sunak (w>=1),sunak (w>=2),starmer (w>=1),starmer (w>=2),burnham (w>=1),burnham (w>=2)
nodes,638,638,643,643,605,605
edges,35739,13486,59528,23196,8100,1358
total weight,67796,45543,120422,84090,10119,3377
density,0.0879,0.0332,0.1442,0.0562,0.0222,0.0037
strongly connected,False,False,True,False,True,False
weakly connected,True,False,True,False,True,False
strong components,2,21,1,7,1,267
largest strong component,637,618,643,637,605,333
weak components,1,13,1,5,1,156
largest weak component,638,626,643,639,605,448


### Who is outside the main component

In [36]:
def outside_core(G):
    core = max(nx.strongly_connected_components(G), key=len)
    rows = []
    for n in G.nodes:
        if n in core:
            continue
        rows.append({
            "name": G.nodes[n]["name"],
            "party": G.nodes[n]["party"],
            "in (they spoke after someone)": G.in_degree(n, weight="weight"),
            "out (someone spoke after them)": G.out_degree(n, weight="weight"),
        })
    return pd.DataFrame(rows)

for key in [(p, 1) for p in PERIODS]:
    out = outside_core(graphs[key])
    print(f"--- {key[0]}, all edges: {len(out)} MPs outside the largest strong component")
    print(out.to_string(index=False) if len(out) else "(none)")

--- sunak, all edges: 1 MPs outside the largest strong component
          name        party  in (they spoke after someone)  out (someone spoke after them)
Rebecca Harris Conservative                              0                               1
--- starmer, all edges: 0 MPs outside the largest strong component
(none)
--- burnham, all edges: 0 MPs outside the largest strong component
(none)


In [37]:
for key in [(p, 2) for p in PERIODS]:
    out = outside_core(graphs[key])
    print(f"--- {key[0]}, weight>=2: {len(out)} MPs outside the largest strong component")
    print("isolated (no repeated exchange):", (out["in (they spoke after someone)"] + out["out (someone spoke after them)"] == 0).sum())
    print("by party:")
    print(out["party"].value_counts().to_string())

--- sunak, weight>=2: 20 MPs outside the largest strong component
isolated (no repeated exchange): 12
by party:
party
Conservative    12
Labour           5
Independent      3
--- starmer, weight>=2: 6 MPs outside the largest strong component
isolated (no repeated exchange): 4
by party:
party
Labour          5
Conservative    1
--- burnham, weight>=2: 272 MPs outside the largest strong component
isolated (no repeated exchange): 153
by party:
party
Labour                     190
Conservative                34
Liberal Democrat            26
Independent                  6
Reform UK                    4
Green                        4
Plaid Cymru                  3
Scottish National Party      2
DUP                          1
Your Party                   1
Restore Britain              1


### Diameter and radius

On the largest strongly connected component, in steps. Weights are counts, not distances, so they're not used here.

In [38]:
def distances(G):
    core = G.subgraph(max(nx.strongly_connected_components(G), key=len))
    ecc = nx.eccentricity(core)
    radius, diameter = min(ecc.values()), max(ecc.values())
    return {
        "component size": core.number_of_nodes(),
        "diameter": diameter,
        "radius": radius,
        "average shortest path": round(nx.average_shortest_path_length(core), 2),
        "center size": sum(1 for e in ecc.values() if e == radius),
        "periphery size": sum(1 for e in ecc.values() if e == diameter),
    }, ecc

dist_rows, eccentricities = {}, {}
for key, G in graphs.items():
    dist_rows[f"{key[0]} (w>={key[1]})"], eccentricities[key] = distances(G)

dist_table = pd.DataFrame(dist_rows)
dist_table

,sunak (w>=1),sunak (w>=2),starmer (w>=1),starmer (w>=2),burnham (w>=1),burnham (w>=2)
component size,637.00,618.00,643.00,637.00,605.0,333.00
diameter,4.00,5.00,4.00,5.00,6.0,14.00
radius,2.00,3.00,2.00,3.00,3.0,5.00
average shortest path,1.98,2.32,1.87,2.08,2.6,3.88
center size,1.00,99.00,7.00,449.00,4.0,1.00
periphery size,42.00,7.00,4.00,1.00,4.0,1.00


Center of each graph (eccentricity = radius)

In [39]:
for key in [(p, 1) for p in PERIODS]:
    ecc = eccentricities[key]
    radius = min(ecc.values())
    center = sorted(names[n] for n, e in ecc.items() if e == radius)
    print(f"--- {key[0]}: {len(center)} MPs at eccentricity {radius}")
    print(", ".join(center[:40]) + (" ..." if len(center) > 40 else ""))

--- sunak: 1 MPs at eccentricity 2
Jim Shannon
--- starmer: 7 MPs at eccentricity 2
Andrew Slaughter, Catherine Atkinson, Chris McDonald, Dan Jarvis, Graham Stuart, Justin Madders, Keir Starmer
--- burnham: 4 MPs at eccentricity 3
Alan Campbell, Andy Burnham, Angela Rayner, Jim McMahon


### Save

In [40]:
summary = pd.concat([table, dist_table])
summary.to_csv(RESULTS_DIR / "structure.csv")
print("saved", RESULTS_DIR / "structure.csv")
summary

saved results/structure.csv


,sunak (w>=1),sunak (w>=2),starmer (w>=1),starmer (w>=2),burnham (w>=1),burnham (w>=2)
nodes,638,638,643,643,605,605
edges,35739,13486,59528,23196,8100,1358
total weight,67796,45543,120422,84090,10119,3377
density,0.0879,0.0332,0.1442,0.0562,0.0222,0.0037
strongly connected,False,False,True,False,True,False
weakly connected,True,False,True,False,True,False
strong components,2,21,1,7,1,267
largest strong component,637,618,643,637,605,333
weak components,1,13,1,5,1,156
largest weak component,638,626,643,639,605,448
